# Firearms Import Control Cross-Referencing and Matching System

> **Sanitized public version.** This notebook documents the analytical workflow and structure of the project. In line with the Responsible Data Note in the project README, it does not include the original data, real file names, field labels tied to the source, specific record counts, named entities, or source-specific operational rules. All cell outputs have been cleared. The code is illustrative of the approach, not runnable against real data.

In [ ]:
# Import libraries to be used
import pandas as pd
import re

# DB A - DOCUMENTATION - df_doc

Import documentation database as df_doc_original, with index = column '_record_id' and forcing SerialNumber column to 'str' dtype (needed to avoid losing '0' on the left and including alphanumeric patterns).

In [ ]:
# Import
df_doc_original = pd.read_excel('documentation_dataset.xlsx',
                                 dtype={'SerialNumber': str},
                                index_col='_record_id')
df_doc_original

In [ ]:
# Check information about dtypes and null values
# 'SerialNumber must' be 'str' (object)
# 'Date Documented' must be datetime

df_doc_original.info()

In [ ]:
# Create a copy of df_doc to preserve the original
df_doc = df_doc_original.copy()

In [ ]:
# Create function to clean all column names
def clean_column_name(col):
    """
    Clean a column name by converting it to lowercase, replacing
    non-alphanumeric characters with '_', and removing consecutive underscores.

    Args:
        col (str): The column name to be cleaned.

    Returns:
        str: The cleaned column name.
    """
    col = col.lower()  # Convert to lowercase
    col = re.sub(r'\W+', '_', col)  # Replace non-alphanumeric characters with '_'
    col = col.replace('___', '_')
    return col

In [ ]:
# Apply clean_column_name function and rename df_doc columns
df_doc.columns = [clean_column_name(col) for col in df_doc.columns]
df_doc.columns

In [ ]:
# Rename and simplify specific columns that will be intensively used in the cross-reference effort
df_doc = df_doc.rename(columns={'government_mark': 'gvtmk',
                                'serial_number': 'sn',
                                'date_documented': 'date_doc',
                                'generic_category_2': 'category',
                                'model_system_designation': 'model',
                                'year_of_manufacture': 'yom',
                                'manufacturing_country': 'mfr_country'})

df_doc.columns

In [ ]:
# Create a DF's copy and duplicate specified columns to preserve original data (sufix '_original')
df_doc = df_doc.copy()
columns_to_copy = ['sn', 'gvtmk', 'category', 'model',
                   'calibre', 'date_doc', 'yom', 'mfr_country']
for column in columns_to_copy:
    df_doc[column + '_original'] = df_doc[column]
df_doc.info()

In [ ]:
# Replace NaN values with 'NA'
columns_na = ['sn', 'category', 'model', 'calibre', 'gvtmk',
              'date_doc', 'mfr_country']
for column in columns_na:
    df_doc[column] = df_doc[column].fillna('NA')

In [ ]:
# Replace null values with 0
columns_to_zero = ['yom', 'yom_original']
for column in columns_to_zero:
    df_doc[column] = df_doc[column].fillna(0)

In [ ]:
# Force columns ['yom', 'yom_original'] to int
columns_to_int = ['yom', 'yom_original']
for column in columns_to_int:
    df_doc[column] = df_doc[column].astype(int)

# Check
df_doc['yom'].dtype

In [ ]:
# Force columns ['sn', 'gvtmk', 'category', 'model', 'calibre', 'mfr_country'] to 'str' (object)
columns_to_str = ['sn', 'gvtmk', 'category', 'model', 'calibre','mfr_country']
for column in columns_to_str:
    df_doc[column] = df_doc[column].astype(str)

# Check
df_doc['sn'].dtype

**Harmonizing category names**

Acoording to the following table:

| Original                           | Modified for Use in Match |
|------------------------------------|----------------------------|
| Assault rifle                      | assaultrifle               |
| Medium machine gun (MMG)           | mg                         |
| Self-loading pistol                | pistol                     |
| Shoulder-fired rocket launchers    | rpg                        |
| Light machine gun (LMG)            | mg                         |
| Sniper rifles                      | dmr                        |
| Heavy machine gun (HMG)            | hmg                        |
| Anti-materiel rifles (AMR)         | amr                        |
| Self-loading rifle or carbine      | rifle                      |

In [ ]:
# Check
df_doc['category'].value_counts()

In [ ]:
# Create a mapping dictionary
doc_category_mapping = {
    "Assault rifle": "assaultrifle",
    "Assault Rifle": "assaultrifle",
    "Medium machine gun (MMG)": "mg",
    "Self-loading pistol": "pistol",
    "Shoulder-fired rocket launchers": "rpg",
    "Light machine gun (LMG)": "mg",
    "Sniper rifles": "dmr",
    "Heavy machine gun (HMG)": "hmg",
    "Anti-materiel rifles (AMR)": "amr",
    "Self-loading rifle or carbine": "rifle"
}

# Replace the 'category' column values based on the mapping
df_doc['category'] = df_doc['category'].map(doc_category_mapping).fillna(df_doc['category'])

# Display the DataFrame to verify the changes
df_doc['category'].value_counts()


In [ ]:
# Change columns ['sn', 'gvtmk', 'mfr_country'] data to upper case
columns_to_upper = ['sn', 'gvtmk', 'mfr_country']
for column in columns_to_upper:
    df_doc[column] = df_doc[column].str.upper()

In [ ]:
# Change columns ['category', 'model', 'calibre'] data to lower case
columns_to_lower = ['category', 'model', 'calibre']
for column in columns_to_lower:
    df_doc[column] = df_doc[column].str.lower()
df_doc.head(3)

In [ ]:
# Count spaces
count_spaces_per_column = df_doc.map(lambda x: ' ' in str(x)).sum()
print("Number of cells with spaces in each column:")
print(count_spaces_per_column)

In [ ]:
# Delete spaces, '-' and '.' in selected columns
columns_to_edit = ['sn', 'category', 'model', 'calibre', 'gvtmk']

for column in columns_to_edit:
    df_doc[column] = df_doc[column].str.replace(' ', '', regex=True)  # delete spaces
    df_doc[column] = df_doc[column].str.replace('-', '', regex=True) # delete '-'

# Delete '.' in ['model'] column
columns_to_edit_point = ['model']
for column in columns_to_edit_point:
    df_doc[column] = df_doc[column].str.replace('.', '') # delete '.'

# Delete '.' in ['sn'] column
df_doc['sn'] = df_doc['sn'].str.replace('.', '') # delete '.'

# Fill NaN values in ['yom'] and force dtype to int
df_doc['yom'] = df_doc['yom'].fillna(0).astype(int)
df_doc.head(3)

In [ ]:
# Check
df_doc['yom'].value_counts().sort_index()

**Deriving year of manufacture (source-specific rule)**

For a subset of records, year of manufacture was derived from a source-specific pattern in the serial number. The operational details of this rule (model identifiers and decoding table) have been removed from this public version, in line with the Responsible Data Note.

In [ ]:
# A source-specific rule derived year of manufacture from the serial number
# for certain models. The decoding table and model identifiers have been
# removed from this public version (see Responsible Data Note).
#
# for index, row in df_doc.iterrows():
#     if <model matches a known pattern> and row['sn'].isdigit() and len(row['sn']) == 8:
#         df_doc.at[index, 'yom'] = <year derived from serial-number prefix>


In [ ]:
# Check
df_doc['yom'].value_counts().sort_index()

In [ ]:
# Create a new colum 'year_doc' based on info from column 'date_doc'.
# This is needed for quality checks that will avoid that we match consignments received after the date that weapon was documented.

# Change 'date_doc' dtype to 'date'
df_doc['date_doc'] = pd.to_datetime(df_doc['date_doc'])

# Create a new colum 'year_doc' based on info from column 'date_doc'
df_doc['year_doc'] = df_doc['date_doc'].dt.year

df_doc.info()

In [ ]:
# Reorganize columns
df_doc = df_doc[[
    'sn', 'gvtmk', 'category', 'model', 'mfr_country', 'calibre', 'year_doc',
    'yom', 'date_doc'] + [
        col for col in df_doc.columns if col not in [
            'sn', 'gvtmk', 'category', 'model', 'mfr_country',
            'calibre', 'year_doc', 'yom', 'date_doc']]]

In [ ]:
# Final check
df_doc.head(3)

# DB B - IMPORT - df_imp

Unlike the documentation dataset, which already has a unique record identifier, this dataset has no such key. It is therefore necessary to create one.

In [ ]:
# Loade the import database
# Specify only the tab/sheet we need to load/import
sheet_name = 'SHEET_NAME'
df_imp_original = pd.read_excel(
    'importation_dataset.xlsx',
    sheet_name=sheet_name, dtype={'SERIAL NUMBER': str})
#Force SN as str to avoid loosing 0s on the left

# Rename the index to start with 'IMP'
df_imp_original.index = 'IMP' + (df_imp_original.index + 1).astype(str)
df_imp_original.head(3)

In [ ]:
# Copy the original df_imp
df_imp = df_imp_original.copy()
df_imp.columns

In [ ]:
# Rename columns (force lower casee and replace spaces with underscore)
df_imp.columns = [clean_column_name(col) for col in df_imp.columns]

df_imp.columns

In [ ]:
# Rename specific columns
df_imp = df_imp.rename(columns={
    'serial_number': 'sn',
    'government_user_mark': 'gvtmk',
    'manufacturing_country': 'mfr_country',
    'year_of_manufacture': 'yom',
    'model_orig': 'model'})

df_imp.columns

In [ ]:
# Reorganize columns
df_imp = df_imp[[
    'sn', 'gvtmk', 'calibre', 'model',
    'mfr_country', 'category', 'year', 'exporting_state'] + [
        col for col in df_imp.columns if col not in [
            'sn', 'gvtmk', 'calibre', 'model',
    'mfr_country', 'category', 'year', 'exporting_state']
        ]
                ]
df_imp.columns

In [ ]:
# Make a df's copy and duplicate specific columns (sufix '_original') to preserve original data
df_imp = df_imp.copy()
columns_to_copy = ['sn', 'year', 'gvtmk', 'category',
                   'model', 'calibre', 'mfr_country', 'exporting_state']
for column in columns_to_copy:
    df_imp[column + '_original'] = df_imp[column]

df_imp.info()

In [ ]:
# Replace null values in specific columns with 'NA'
columns_na = ['sn', 'gvtmk', 'category', 'model', 'calibre', 'mfr_country', 'exporting_state']
for column in columns_na:
    df_imp[column] = df_imp[column].fillna('NA')

# Replace null values in 'year' column with 0
df_imp['year'] = df_imp['year'].fillna(0).astype(int)

df_imp.head(3)

In [ ]:
# Attribute string format to columns
columns_to_str = ['sn', 'gvtmk', 'category', 'model', 'calibre', 'mfr_country']
for column in columns_to_str:
    df_imp[column] = df_imp[column].astype(str)

df_imp['sn'].dtype

**Harmonizing category names**

Acoording to the following table:

|              Original             |     Modified |
|-----------------------------------|--------------|
| Assault rifle                     | assaultrifle |
| Automatic grenade launcher (AGL)  | agl          |
| Explosives                        | explosives   |
| Grenade launchers                 | agl          |
| Heavy machine gun (HMG)           | hmg          |
| Light Machine Gun                 | mg           |
| Medium machine gun                | mg           |
| Medium machine gun (MMG)          | mg           |
| Night vision monocular            | nvm          |
| Self-loading pistol               | pistol       |
| Self-loading rifle                | rifle        |
| Shoulder-fired rocket launcher    | rpg          |

In [ ]:
# Check original names
df_imp['category'].value_counts()

In [ ]:
# Create a mapping dictionary for category modification
imp_category_mapping = {
    "Assault rifle": "assaultrifle",
    "Automatic grenade launcher (AGL)": "agl",
    "Explosives": "explosives",
    "Grenade launchers": "agl",
    "Heavy machine gun (HMG)": "hmg",
    "Heavy Machine Gun (HMG)": "hmg",
    "Light Machine Gun": "mg",
    "Medium machine gun": "mg",
    "Medium machine gun (MMG)": "mg",
    "Medium Machine gun (MMG)": "mg",
    "Night vision monocular": "nvm",
    "Self-loading pistol": "pistol",
    "Self-loading rifle": "rifle",
    "Shoulder-fired rocket launcher": "rpg"
}

# Replace the 'category' column values based on the mapping
df_imp['category'] = df_imp['category'].map(imp_category_mapping).fillna(df_imp['category'])

# Check changes
df_imp['category'].value_counts()

In [ ]:
# Columns to upper
columns_to_upper = ['sn', 'gvtmk', 'mfr_country', 'exporting_state']
for column in columns_to_upper:
    df_imp[column] = df_imp[column].str.upper()

In [ ]:
# Columns to lower
columns_to_lower = ['category', 'model', 'calibre']
for column in columns_to_lower:
    df_imp[column] = df_imp[column].str.lower()

df_imp.head(3)

In [ ]:
# Count cells with spaces
count_spaces_per_column = df_imp.map(lambda x: ' ' in str(x)).sum()
print("Number of cells with spaces in each column:")
print(count_spaces_per_column)

In [ ]:
# Create new columns after cleaning fields (delete spaces, hyphens, dots, etc.) in selected columns, to preserve the originals
columns_to_edit = ['sn', 'category', 'calibre', 'model', 'gvtmk', 'mfr_country', 'exporting_state']

for column in columns_to_edit:
    df_imp[column] = df_imp[column].str.replace(' ', '', regex=True)  # delete spaces
    df_imp[column] = df_imp[column].str.replace('-', '', regex=True) # delete '-'

#Delete '.' in model
df_imp['model'] = df_imp['model'].str.replace('.', '')

#Delete '.' in sn
df_imp['sn'] = df_imp['sn'].str.replace('.', '')

#Delete '.' in 'exporting_state'
df_imp['exporting_state'] = df_imp['exporting_state'].str.replace('.', '')

df_imp.head(3)

In [ ]:
# Check if specif 'sn' from original DB was correctly imported (with left 0s)
# iloc[] changes acording to the original DB row
df_imp.iloc[10137]['sn']

In [ ]:
# Check
df_imp

# MATCHING

## Copy and prepare  processesd DBs to match

In [ ]:
# Copy processed DBs to use in the matching processes
df_documentation = df_doc.copy()

df_importation = df_imp.copy()

In [ ]:
# Rename columns in df_documentation with sufix _d
df_documentation.columns = df_documentation.columns + '_d'

# Select a specific subset of columns in df_documentation
df_doctomatch = df_documentation[
    ['sn_d', 'gvtmk_d', 'calibre_d', 'category_d', 'mfr_country_d', 'model_d',
     'year_doc_d', 'yom_d']].copy()

# Define subset df_doctomatch index to correspond to df_documentation index
df_doctomatch.index = df_documentation.index

#Create new columns 'match_level' e 'index_i' with standard values, which
#will be completed after processing.
df_doctomatch['match_level_d'] = 'no_match'
df_doctomatch['index_i'] = 'NA'

# Inspect columns
df_doctomatch.info()

In [ ]:
# Rename columns in df_importation with sufix _i
df_importation.columns = df_importation.columns + '_i'

# Select subset from df_importation that will be used in cross-referencing process
df_imptomatch = df_importation[['sn_i', 'year_i', 'gvtmk_i', 'model_i',
                                'calibre_i', 'mfr_country_i', 'category_i', 'exporting_state_i']]

#Define subset df_imptomatch index to correspond to df_importation index
df_imptomatch.index = df_imptomatch.index

# Inspect db
df_imptomatch.info()


In [ ]:
# Create a copy of the subsetx 'tomatch'
df_imp_tomatch = df_imptomatch.copy()
df_doc_tomatch = df_doctomatch.copy()

# Check
df_doc_tomatch

# Matches

* SN Perfect Match (A, B)
* SN Partial Match (C, D, E)
* UniqueGovtMark Perfect Match (F)
* SN 4 digits Perfect Match (E)

This is the heart of the matching process, with the different levels created.

In [ ]:
# Create a function to check match levels
def check_match_level(row_d):
    global df_imp_tomatch
    sn_d, gvtmk_d, category_d, calibre_d, model_d, year_doc_d, yom_d = row_d['sn_d'], row_d['gvtmk_d'], row_d['category_d'], row_d['calibre_d'], row_d['model_d'], row_d['year_doc_d'], row_d['yom_d']

    for index_i, row_i in df_imp_tomatch.iterrows():
        sn_i, year_i, gvtmk_i, category_i, calibre_i, model_i = row_i['sn_i'], row_i['year_i'], row_i['gvtmk_i'], row_i['category_i'], row_i['calibre_i'], row_i['model_i']

        conditions_met = (
            calibre_d == calibre_i and \
                  category_d == category_i and \
                    year_doc_d >= year_i and \
                        yom_d <= year_i)

        if len(sn_d) >= 6 and sn_d == sn_i:
            if gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and model_d == model_i and model_d != 'NA' and conditions_met:
                return 'A1', index_i
            elif gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and conditions_met:
                return 'A2', index_i
            elif (((gvtmk_d == 'NA') and (gvtmk_i != 'NA')) or \
                ((gvtmk_d != 'NA') and (gvtmk_i == 'NA')) or \
                ((gvtmk_d == 'NA') and (gvtmk_i == 'NA'))) and conditions_met:
                    return 'A3', index_i
        elif len(sn_d) == 5 and sn_d == sn_i:
            if gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and model_d == model_i and model_d != 'NA' and conditions_met:
                return 'B1', index_i
            elif gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and conditions_met:
                return 'B2', index_i
            elif (((gvtmk_d == 'NA') and (gvtmk_i != 'NA')) or \
                ((gvtmk_d != 'NA') and (gvtmk_i == 'NA')) or \
                ((gvtmk_d == 'NA') and (gvtmk_i == 'NA'))) and conditions_met:
                    return 'B3', index_i
        elif len(sn_d) >= 6 and len(sn_i) == 6 and sn_d[-6:] == sn_i:
            if gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and model_d == model_i and model_d != 'NA' and conditions_met:
                return 'C1', index_i
            elif gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and conditions_met:
                return 'C2', index_i
            elif (((gvtmk_d == 'NA') and (gvtmk_i != 'NA')) or \
                ((gvtmk_d != 'NA') and (gvtmk_i == 'NA')) or \
                ((gvtmk_d == 'NA') and (gvtmk_i == 'NA'))) and conditions_met:
                    return 'C3', index_i
        elif len(sn_d) >= 5 and len(sn_i) == 5 and sn_d[-5:] == sn_i:
            if gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and model_d == model_i and model_d != 'NA' and conditions_met:
                return 'D1', index_i
            elif gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and conditions_met:
                return 'D2', index_i
            elif (((gvtmk_d == 'NA') and (gvtmk_i != 'NA')) or \
                ((gvtmk_d != 'NA') and (gvtmk_i == 'NA')) or \
                ((gvtmk_d == 'NA') and (gvtmk_i == 'NA'))) and conditions_met:
                    return 'D3', index_i
        elif (model_d == 'modelx' or model_d == 'modelxv' or model_d == 'modelxp'):
            if len(sn_d) == 6 and len(sn_i) == 5 and sn_d[:5] == sn_i:
                if gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and model_d == model_i and model_d != 'NA' and conditions_met:
                    return 'E1', index_i
                elif gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and conditions_met:
                    return 'E2', index_i
                elif (((gvtmk_d == 'NA') and (gvtmk_i != 'NA')) or \
                ((gvtmk_d != 'NA') and (gvtmk_i == 'NA')) or \
                ((gvtmk_d == 'NA') and (gvtmk_i == 'NA'))) and conditions_met:
                    return 'E3', index_i
        elif (len(gvtmk_d) >= 11) and (gvtmk_d == gvtmk_i):
            if model_d == model_i and model_d != 'NA' and conditions_met:
                return 'F1', index_i
            elif conditions_met:
                return 'F2', index_i
        elif len(sn_d) == 4 and len(sn_i) == 4 and sn_d == sn_i:
            if gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and model_d == model_i and model_d != 'NA' and conditions_met:
                return 'G1', index_i
            elif gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and conditions_met:
                return 'G2', index_i
            elif model_d == model_i and model_d != 'NA' and conditions_met:
                return 'G3', index_i

    return 'no_match', 'NA'

In [ ]:
# Apply def check_match_level on the DBs
df_doc_tomatch[['match_level_d', 'index_i']] = df_doc_tomatch.apply(check_match_level, axis=1, result_type='expand')

In [ ]:
# Check matches
df_doc_tomatch['match_level_d'].value_counts()

In [ ]:
# Create a copy of the matechd df
df_doc_tomatch2 = df_doc_tomatch.copy()

In [ ]:
# Export first matched df to excel as 'matched_intermediate.xlsx'
df_doc_tomatch.to_excel('matched_intermediate.xlsx', index=True)

In [ ]:
# Create a dictionary with excluded indices in df_imp_tomatch which had already matched
excluded_index_i = []

for index, row in df_doc_tomatch2.iterrows():
    # Verifica se o valor de index_i é um índice válido (não 'NA' ou similar)
    if row['index_i'] != 'NA':
        # Adiciona o valor de index_i à lista do dicionário para o índice atual
        excluded_index_i.append(row['index_i'])


In [ ]:
# Check
excluded_index_i

In [ ]:
# Create df_imp_tomatch_additional excluding rows with index in excluded_index_i
df_imp_tomatch_additional = df_imp_tomatch.drop(excluded_index_i)

In [ ]:
# Initialize the columns 'match_level_d_additional' and 'index_i_additional' with default values
df_doc_tomatch2['match_level_d_additional'] = 'no_match'
df_doc_tomatch2['index_i_additional'] = 'NA'

In [ ]:
def find_additional_matches(row_d):
    """
    This function finds additional matches for a given row in the documentation dataset
    by comparing it with rows in the import dataset. It returns a match level and the 
    index of the matching row in the import dataset.

    Parameters:
    row_d (pd.Series): A row from the documentation dataset.

    Returns:
    tuple: A tuple containing the match level and the index of the matching row in the import dataset.
    """
    global df_imp_tomatch_additional
    sn_d, gvtmk_d, category_d, calibre_d, model_d, year_doc_d, yom_d = row_d['sn_d'], row_d['gvtmk_d'], row_d['category_d'], row_d['calibre_d'], row_d['model_d'], row_d['year_doc_d'], row_d['yom_d']

    for index_i, row_i in df_imp_tomatch_additional.iterrows():
        sn_i, year_i, gvtmk_i, category_i, calibre_i, model_i = row_i['sn_i'], row_i['year_i'], row_i['gvtmk_i'], row_i['category_i'], row_i['calibre_i'], row_i['model_i']

        # Check if the basic conditions are met
        conditions_met = (
            calibre_d == calibre_i and \
            category_d == category_i and \
            year_doc_d >= year_i and \
            yom_d <= year_i
        )

        # Check for exact serial number match with length >= 6
        if len(sn_d) >= 6 and sn_d == sn_i:
            if gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and model_d == model_i and model_d != 'NA' and conditions_met:
                return 'A1', index_i
            elif gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and conditions_met:
                return 'A2', index_i
            elif (((gvtmk_d == 'NA') and (gvtmk_i != 'NA')) or \
                  ((gvtmk_d != 'NA') and (gvtmk_i == 'NA')) or \
                  ((gvtmk_d == 'NA') and (gvtmk_i == 'NA'))) and conditions_met:
                return 'A3', index_i

        # Check for exact serial number match with length == 5
        elif len(sn_d) == 5 and sn_d == sn_i:
            if gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and model_d == model_i and model_d != 'NA' and conditions_met:
                return 'B1', index_i
            elif gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and conditions_met:
                return 'B2', index_i
            elif (((gvtmk_d == 'NA') and (gvtmk_i != 'NA')) or \
                  ((gvtmk_d != 'NA') and (gvtmk_i == 'NA')) or \
                  ((gvtmk_d == 'NA') and (gvtmk_i == 'NA'))) and conditions_met:
                return 'B3', index_i

        # Check for partial serial number match with length >= 6
        elif len(sn_d) >= 6 and len(sn_i) == 6 and sn_d[-6:] == sn_i:
            if gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and model_d == model_i and model_d != 'NA' and conditions_met:
                return 'C1', index_i
            elif gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and conditions_met:
                return 'C2', index_i
            elif (((gvtmk_d == 'NA') and (gvtmk_i != 'NA')) or \
                  ((gvtmk_d != 'NA') and (gvtmk_i == 'NA')) or \
                  ((gvtmk_d == 'NA') and (gvtmk_i == 'NA'))) and conditions_met:
                return 'C3', index_i

        # Check for partial serial number match with length == 5
        elif len(sn_d) >= 5 and len(sn_i) == 5 and sn_d[-5:] == sn_i:
            if gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and model_d == model_i and model_d != 'NA' and conditions_met:
                return 'D1', index_i
            elif gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and conditions_met:
                return 'D2', index_i
            elif (((gvtmk_d == 'NA') and (gvtmk_i != 'NA')) or \
                  ((gvtmk_d != 'NA') and (gvtmk_i == 'NA')) or \
                  ((gvtmk_d == 'NA') and (gvtmk_i == 'NA'))) and conditions_met:
                return 'D3', index_i

        # Check for specific models with partial serial number match
        elif (model_d == 'modelx' or model_d == 'modelxv' or model_d == 'modelxp'):
            if len(sn_d) == 6 and len(sn_i) == 5 and sn_d[:5] == sn_i:
                if gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and model_d == model_i and model_d != 'NA' and conditions_met:
                    return 'E1', index_i
                elif gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and conditions_met:
                    return 'E2', index_i
                elif (((gvtmk_d == 'NA') and (gvtmk_i != 'NA')) or \
                      ((gvtmk_d != 'NA') and (gvtmk_i == 'NA')) or \
                      ((gvtmk_d == 'NA') and (gvtmk_i == 'NA'))) and conditions_met:
                    return 'E3', index_i

        # Check for exact government mark match with length >= 11
        elif (len(gvtmk_d) >= 11) and (gvtmk_d == gvtmk_i):
            if model_d == model_i and model_d != 'NA' and conditions_met:
                return 'F1', index_i
            elif conditions_met:
                return 'F2', index_i

        # Check for exact serial number match with length == 4
        elif len(sn_d) == 4 and len(sn_i) == 4 and sn_d == sn_i:
            if gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and model_d == model_i and model_d != 'NA' and conditions_met:
                return 'G1', index_i
            elif gvtmk_d == gvtmk_i and gvtmk_d != 'NA' and conditions_met:
                return 'G2', index_i
            elif model_d == model_i and model_d != 'NA' and conditions_met:
                return 'G3', index_i

    # Return 'no_match' if no conditions are met
    return 'no_match', 'NA'

In [ ]:
for index, row in df_doc_tomatch2.iterrows():
    # Call find_additional_matches for the current row
    # Note that df_imp_tomatch_additional no longer contains the excluded indices
    match_level_d_additional, index_i_additional = find_additional_matches(row)
    
    # Update df_doc_tomatch2 with the results
    df_doc_tomatch2.at[index, 'match_level_d_additional'] = match_level_d_additional
    df_doc_tomatch2.at[index, 'index_i_additional'] = index_i_additional
    
    # Update excluded_index_i to include the new match, if any
    if index_i_additional != 'NA':
        excluded_index_i.append(index_i_additional)
        # It is also important to update df_imp_tomatch_additional to reflect this new exclusion
        df_imp_tomatch_additional = df_imp_tomatch_additional.drop(index_i_additional)


In [ ]:
# Check match levels in df_doc_tomatch2
df_doc_tomatch2['match_level_d_additional'].value_counts()

In [ ]:
# Count the number of occurrences of each match level in descending order
match_counting = df_doc_tomatch2['match_level_d'].value_counts()
display(match_counting)

In [ ]:
def postmatches_model_check(row_d):
    """
    This function checks for errors in the match level for rows in the documentation dataset
    that belong to the 'assaultrifle' category and were exported from specific flagged states (names removed in the public version).
    It iterates over the import dataset to verify if the match level and index should remain unchanged.

    Parameters:
    row_d (pd.Series): A row from the documentation dataset.

    Returns:
    tuple: A tuple containing the match level and the index of the matching row in the import dataset.
           If no conditions are met, it returns ('no_match', 'NA').
    """
    global df_imp_tomatch
    category_d, model_d, match_level_d, index_i = row_d['category_d'], row_d['model_d'], row_d['match_level_d'], row_d['index_i']

    # Iterate over df_imp_tomatch with iterrows()
    for index, row_i in df_imp_tomatch.iterrows():
        model_i, exporting_state_i = row_i['model_i'], row_i['exporting_state_i']

        if (match_level_d not in ['no_match', 'A1', 'B1', 'C1', 'D1', 'F1', 'G1', 'G3']) and \
           (category_d == 'assaultrifle') and \
           (exporting_state_i == 'COUNTRY_A' or exporting_state_i == 'COUNTRY_B') and \
           model_d == model_i:
            # If conditions are met, return the current values unchanged.
            return match_level_d, index_i

    # If no conditions are met, return 'no_match', 'NA'.
    return 'no_match', 'NA'

In [ ]:
# Apply function postmatches_model_check to check for errors in category assaultrifle
df_doc_tomatch2[['match_level_d', 'index_i']] = df_doc_tomatch2.apply(postmatches_model_check, axis=1, result_type='expand')

In [ ]:
def postmatches_model_check_additional(row_d):
    """
    This function checks for errors in the additional match level for rows in the documentation dataset
    that belong to the 'assaultrifle' category and were exported from specific flagged states (names removed in the public version).
    It iterates over the import dataset to verify if the additional match level and index should remain unchanged.

    Parameters:
    row_d (pd.Series): A row from the documentation dataset.

    Returns:
    tuple: A tuple containing the additional match level and the index of the matching row in the import dataset.
           If no conditions are met, it returns ('no_match', 'NA').
    """
    global df_imp_tomatch
    category_d, model_d, match_level_d_additional, index_i_additional = row_d['category_d'], row_d['model_d'], row_d['match_level_d_additional'], row_d['index_i_additional']

    # Iterate over df_imp_tomatch with iterrows()
    for index, row_i in df_imp_tomatch.iterrows():
        model_i, exporting_state_i = row_i['model_i'], row_i['exporting_state_i']

        if (match_level_d_additional not in ['no_match', 'A1', 'B1', 'C1', 'D1', 'F1', 'G1', 'G3']) and \
           (category_d == 'assaultrifle') and \
           (exporting_state_i == 'COUNTRY_A' or exporting_state_i == 'COUNTRY_B') and \
           model_d == model_i:
            # If conditions are met, return the current values unchanged.
            return match_level_d_additional, index_i_additional

    # If no conditions are met, return 'no_match', 'NA'.
    return 'no_match', 'NA'

In [ ]:
# Apply function postmatches_model_check to check for errors in category assaultrifle
df_doc_tomatch2[['match_level_d_additional', 'index_i_additional']] = df_doc_tomatch2.apply(postmatches_model_check, axis=1, result_type='expand')

In [ ]:
# Check df_documentation columns before combining information with df_doctomatch after matching
df_documentation.columns

In [ ]:
# Create a column list, with 'match_level' and 'index_i' at the begining
cols = ['match_level_d', 'index_i', 'match_level_d_additional', 'index_i_additional'] + [
    col for col in df_doc_tomatch2.columns if col not in [
        'match_level_d', 'index_i', 'match_level_d_additional', 'index_i_additional']]

# Reorder the columns of df_a using the modified column list
df_doc_tomatch2 = df_doc_tomatch2[cols]
df_doc_tomatch2

In [ ]:
# Create a new DF merging df_doc_tomatch with df_documentation using the unique reference/key _record_id_
df_documentation_matches = pd.merge(df_doc_tomatch2, df_documentation.drop(
    ['sn_d', 'year_doc_d', 'gvtmk_d', 'calibre_d', 'category_d', 'model_d',
     'yom_d','mfr_country_d', 'security_force_d','security_force2_d',
     'region_d', 'district_d', 'generic_category_1_d','mfr_country_d',
     'end_user_indicated_d', 'sn_original_d', 'gvtmk_original_d',
     'category_original_d', 'model_original_d', 'calibre_original_d',
    'date_doc_original_d'],
    axis=1), left_index=True, right_index=True)

df_documentation_matches.info()

In [ ]:
# Display the 'date_doc_d' column from the df_documentation_matches DataFrame
df_documentation_matches['date_doc_d']

In [ ]:
# Prepare df_importation to merge with df_documentation_matches
df_importation_to_merge = df_importation.drop(
    ['weapon_number_i', 'batch_lot_number_i',
     'quantity_in_box_crate_i', 'expiry_date__i', 'box_number_i',
     'category_original_i','model_original_i',
     'calibre_original_i', 'mfr_country_original_i', 'exporting_state_original_i'], axis=1)

df_importation_to_merge['index_importation'] = df_importation_to_merge.index

df_importation_to_merge

In [ ]:
# Perform the first merge using 'index_i'
df_documentation_matches_final = df_documentation_matches.merge(
    df_importation_to_merge, left_on='index_i', right_index=True, how='left')

# Perform the second merge directly on df_documentation_matches_final using 'index_i_additional'
# Use suffixes to differentiate the columns added by the second merge
df_documentation_matches_final = df_documentation_matches_final.merge(
    df_importation_to_merge, left_on='index_i_additional', right_index=True, how='left', suffixes=('', '_additional'))

In [ ]:
# Check df_documentation_matches_final columns
df_documentation_matches_final.columns

In [ ]:
# Reorder columns
cols = [
    "source_d",
    "status_d",
    "match_level_d",
    "match_level_d_additional",
    "sn_d",
    "gvtmk_d",
    "category_d",
    'mfr_country_d',
    "model_d",
    "calibre_d",
    "year_doc_d",
    "yom_d",
    "yom_original_d",
    "date_doc_d",
    "year_imported_d",
    "exporting_state_d",
    "index_i",
    "sn_i",
    "gvtmk_i",
    "category_i",
    'mfr_country_i',
    "model_i",
    "calibre_i",
    "yom_i",
    "exporting_state_i",
    "year_i",
    "index_i_additional",
    "sn_i_additional",
    "gvtmk_i_additional",
    "category_i_additional",
    'mfr_country_i_additional',
    "model_i_additional",
    "calibre_i_additional",
    "yom_i_additional",
    "exporting_state_i_additional",
    "year_i_additional"
]

df_documentation_matches_final = df_documentation_matches_final[cols]

# Check
df_documentation_matches_final

In [ ]:
# Check final df
df_documentation_matches_final.info()

In [ ]:
# Export final df to .xlsx
df_documentation_matches_final.to_excel(
    'matched_dataset_final.xlsx', index=True)

In [ ]:
# Export final df to .csv
df_documentation_matches_final.to_csv(
    'matched_dataset_final.csv', index=True)